In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd
from bs4 import BeautifulSoup

# Setup Chrome options
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

# Corrected tender data extractor
def extract_tender_data(card):
    try:
        html = card.get_attribute("innerHTML")
        soup = BeautifulSoup(html, "html.parser")

        title_tag = soup.find("a", class_="m-brief")
        tdr_tag = soup.select_one(".m-brief .m-tender-id")
        month = soup.select_one(".month")
        day = soup.select_one(".day")
        year = soup.select_one(".year")
        value = soup.select_one(".tender-value")

        tdr = tdr_tag.text.strip() if tdr_tag else None
        if not tdr or not tdr.isdigit():
            return None

        due_date = f"{month.text.strip()} {day.text.strip()}, {year.text.strip()}" if month and day and year else "N/A"

        return {
            "TDR": tdr,
            "Title": title_tag['title'].strip() if title_tag else "N/A",
            "Due Date": due_date,
            "Tender Value": value.get_text(strip=True).replace("\n", " ") if value else "N/A",
            "URL": title_tag.get("href").strip() if title_tag else "N/A"
        }

    except Exception as e:
        print(f"⚠️ Error parsing tender card: {e}")
        return None

# Base pagination URL
base_url = "https://www.tenderdetail.com/Tenders/Result.aspx?type=domestic&kw=amrut&p="

all_tenders = []
seen_ids = set()

# Iterate pages until no tenders are found
for page in range(1, 1000):  # Adjust max page as needed
    url = base_url + str(page)
    print(f"🌐 Visiting: {url}")
    driver.get(url)
    time.sleep(3)

    cards = driver.find_elements(By.CSS_SELECTOR, ".tender_row")
    print(f"📄 Page {page}: {len(cards)} tenders found")

    if not cards:
        print("✅ No tenders found, stopping pagination.")
        break

    for card in cards:
        tender = extract_tender_data(card)
        if tender and tender["TDR"] not in seen_ids:
            seen_ids.add(tender["TDR"])
            all_tenders.append(tender)
        elif not tender:
            print("⚠️ Skipped a card due to missing TDR")

print(f"✅ Total tenders scraped: {len(all_tenders)}")

# Save to Excel
df = pd.DataFrame(all_tenders)
df.to_excel("amrut_tenders_final.xlsx", index=False)
print("📁 Saved all tenders to 'amrut_tenders_final.xlsx'")

driver.quit()


🌐 Visiting: https://www.tenderdetail.com/Tenders/Result.aspx?type=domestic&kw=amrut&p=1
📄 Page 1: 10 tenders found
🌐 Visiting: https://www.tenderdetail.com/Tenders/Result.aspx?type=domestic&kw=amrut&p=2


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=137.0.7151.104)
Stacktrace:
	GetHandleVerifier [0x0x7ff6871efe75+79173]
	GetHandleVerifier [0x0x7ff6871efed0+79264]
	(No symbol) [0x0x7ff686fa9e5a]
	(No symbol) [0x0x7ff686f820f1]
	(No symbol) [0x0x7ff68703017e]
	(No symbol) [0x0x7ff687050922]
	(No symbol) [0x0x7ff687028743]
	(No symbol) [0x0x7ff686ff14c1]
	(No symbol) [0x0x7ff686ff2253]
	GetHandleVerifier [0x0x7ff6874ba2ad+3004797]
	GetHandleVerifier [0x0x7ff6874b46fd+2981325]
	GetHandleVerifier [0x0x7ff6874d3350+3107360]
	GetHandleVerifier [0x0x7ff68720a9fe+188622]
	GetHandleVerifier [0x0x7ff68721228f+219487]
	GetHandleVerifier [0x0x7ff6871f8dc4+115860]
	GetHandleVerifier [0x0x7ff6871f8f79+116297]
	GetHandleVerifier [0x0x7ff6871df528+11256]
	BaseThreadInitThunk [0x0x7ffc413bdbe7+23]
	RtlUserThreadStart [0x0x7ffc420e5a4c+44]


In [ ]:
import os
import pandas as pd
# Convert to DataFrame and save to known location (e.g., Desktop)
df = pd.DataFrame(all_tenders)

# Update this path as needed to match your actual desktop folder
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "amrut_tenders_final.xlsx")

df.to_excel(desktop_path, index=False)
print(f"📁 Saved all tenders to: {desktop_path}")
